# Green-List Distribution Analysis: Human vs AI-Generated Code

Analyzing how green-list letter frequencies compare across human-authored and AI-generated (baseline & watermarked) code.

In [ ]:
import ast
import hashlib
import json
import random
import string
import builtins
import keyword
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import rel_entr

ROOT = Path("../../")
DATASET_DIR = ROOT / "datasets"
OUTPUT_DIR = ROOT / "output"

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

## Green/Red Set Generation (from `shared_utils.py`)

In [ ]:
def get_frequent_candidates(top_n=18):
    """Load the top-N frequent initial characters from the pre-computed frequency files."""
    freq_sum = defaultdict(int)
    for path in [
        OUTPUT_DIR / "dataset_analysis" / "humaneval_letter_freqs.json",
        OUTPUT_DIR / "dataset_analysis" / "mbpp_letter_freqs.json",
    ]:
        import ast as _ast
        with open(path) as f:
            data = _ast.literal_eval(f.read())
            for char, count in data["letter_freqs"].items():
                freq_sum[char] += count
    sorted_chars = sorted(freq_sum.items(), key=lambda x: x[1], reverse=True)
    return [char for char, _ in sorted_chars[:top_n]]


def build_green_set(secret_key, candidate_letters, g_min=8, g_max=18):
    """Deterministic green-set construction using SHA-256."""
    seed_hash = int(hashlib.sha256(secret_key.encode()).hexdigest(), 16)
    green_set_size = g_min + (seed_hash % (g_max - g_min + 1))
    rng = random.Random(seed_hash)
    shuffled = list(candidate_letters)
    rng.shuffle(shuffled)
    green_set = set(shuffled[:green_set_size])
    red_set = set(candidate_letters) - green_set
    return green_set, red_set, green_set_size


CANDIDATES = get_frequent_candidates(18)
print(f"Candidate pool ({len(CANDIDATES)} chars): {CANDIDATES}")

In [ ]:
SECRET_KEYS = [
    "watermark-key-alpha",
    "watermark-key-beta",
    "watermark-key-gamma",
    "watermark-key-delta",
    "watermark-key-epsilon",
    "research-secret-2024",
    "prompt-mark-test-key",
    "crypto-wm-seed-001",
]

key_sets = {}
for key in SECRET_KEYS:
    g, r, sz = build_green_set(key, CANDIDATES)
    key_sets[key] = {"green": g, "red": r, "size": sz}
    print(f"Key '{key}': |G|={sz}, green={sorted(g)}")

## Identifier Extraction

In [ ]:
COMMON_STD = {
    "self", "re", "cls", "append", "join", "dummy_function", "find", "kwargs",
    "args", "range", "print", "len", "dict", "list", "str", "int", "float",
    "set", "tuple", "os", "np", "subprocess", "now", "today", "timedelta",
    "strptime", "date", "time", "datetime", "logging", "log", "info", "debug",
    "error", "warning", "exception", "lower", "upper", "strip", "split",
    "replace", "endswith", "startswith", "extend", "insert", "remove", "pop",
    "sort", "clear", "keys", "values", "items", "get", "update", "copy",
    "format", "count", "index",
}
SKIP = set(dir(builtins)).union(COMMON_STD).union(set(keyword.kwlist))


def extract_identifiers(code: str) -> list[str]:
    ids = []

    class V(ast.NodeVisitor):
        def visit_Name(self, n):
            if n.id not in SKIP: ids.append(n.id)
            self.generic_visit(n)

        def visit_FunctionDef(self, n):
            if n.name not in SKIP: ids.append(n.name)
            for a in n.args.args:
                if a.arg not in SKIP: ids.append(a.arg)
            self.generic_visit(n)

        visit_AsyncFunctionDef = visit_FunctionDef

        def visit_ClassDef(self, n):
            if n.name not in SKIP: ids.append(n.name)
            self.generic_visit(n)

        def visit_Import(self, n): pass
        def visit_ImportFrom(self, n): pass

    try:
        V().visit(ast.parse(code))
    except SyntaxError:
        pass
    return ids


def initial_char_counter(codes: list[str]) -> Counter:
    c = Counter()
    for code in codes:
        for ident in extract_identifiers(code):
            if ident and ident[0].isalpha():
                c[ident[0].lower()] += 1
    return c

## Load All Code Sources

In [ ]:
# Human-authored datasets
humaneval_df = pd.read_parquet(DATASET_DIR / "human_eval_164.parquet")
humaneval_codes = (humaneval_df['prompt'] + humaneval_df['canonical_solution']).tolist()

mbpp_records = [json.loads(l) for l in open(DATASET_DIR / "sanitized-mbpp-raw.jsonl")]
mbpp_codes = [r['code'] for r in mbpp_records]

classeval_df = pd.read_parquet(DATASET_DIR / "classEval.parquet")
classeval_codes = classeval_df['solution_code'].tolist()

human_sources = {
    "HumanEval": humaneval_codes,
    "MBPP": mbpp_codes,
    "ClassEval": classeval_codes,
}

# AI-generated code
def load_py_folder(folder):
    return [p.read_text() for p in sorted(folder.glob("*.py"))]

# Watermarked experiment folders (exclude expX = baseline)
wm_folders = sorted([
    d for d in OUTPUT_DIR.iterdir()
    if d.is_dir() and d.name.startswith("claude_exp") and "expX" not in d.name
])
baseline_folders = sorted([
    d for d in OUTPUT_DIR.iterdir()
    if d.is_dir() and d.name.startswith("claude_expX")
])

# Combine all watermarked code
wm_codes = []
for f in wm_folders:
    wm_codes.extend(load_py_folder(f))

# Combine all baseline code
baseline_codes = []
for f in baseline_folders:
    baseline_codes.extend(load_py_folder(f))

print(f"Human: HumanEval={len(humaneval_codes)}, MBPP={len(mbpp_codes)}, ClassEval={len(classeval_codes)}")
print(f"AI Baseline (no watermark): {len(baseline_codes)} files")
print(f"AI Watermarked: {len(wm_codes)} files from {len(wm_folders)} experiments")

In [ ]:
# Pre-compute initial character counters for all sources
counters = {}
for name, codes in human_sources.items():
    counters[name] = initial_char_counter(codes)
    print(f"{name}: {sum(counters[name].values())} identifiers")

counters["AI Baseline"] = initial_char_counter(baseline_codes)
print(f"AI Baseline: {sum(counters['AI Baseline'].values())} identifiers")

counters["AI Watermarked"] = initial_char_counter(wm_codes)
print(f"AI Watermarked: {sum(counters['AI Watermarked'].values())} identifiers")

## Green-Letter Frequency: Human vs AI (Pairwise)

In [ ]:
def to_prob(counter):
    total = sum(counter.values())
    return {k: v / total for k, v in counter.items()} if total else {}


def green_ratio(counter, green_set):
    """Fraction of identifiers whose initial character is in the green set."""
    total = sum(counter.values())
    if total == 0:
        return 0.0
    return sum(counter.get(c, 0) for c in green_set) / total

In [ ]:
# For each secret key, compute green-letter ratios across all sources
ratio_rows = []
for key, info in key_sets.items():
    g = info["green"]
    row = {"key": key, "|G|": info["size"]}
    for src_name, ctr in counters.items():
        row[src_name] = green_ratio(ctr, g)
    ratio_rows.append(row)

ratio_df = pd.DataFrame(ratio_rows).set_index("key")
print("Green-letter ratio (fraction of identifiers starting with a green letter):")
ratio_df.round(4)

In [ ]:
# Pairwise plots: AI row (Baseline + Watermarked) on top, Human dataset on bottom
human_ds_names = ["HumanEval", "MBPP", "ClassEval"]

for key_name, info in list(key_sets.items())[:4]:  # show 4 keys
    green = sorted(info["green"])
    
    fig, axes = plt.subplots(len(human_ds_names), 2, figsize=(14, 3.2 * len(human_ds_names)),
                              sharex=True)
    fig.suptitle(f"Key: {key_name}  |  Green set ({info['size']}): {green}", fontsize=13, y=1.01)

    x = np.arange(len(green))
    width = 0.35

    for row_idx, human_name in enumerate(human_ds_names):
        h_probs = to_prob(counters[human_name])
        b_probs = to_prob(counters["AI Baseline"])
        w_probs = to_prob(counters["AI Watermarked"])

        # Left panel: absolute probability of each green letter
        ax_ai = axes[row_idx, 0]
        ax_hu = axes[row_idx, 1]

        # AI side (Baseline vs Watermarked)
        b_vals = [b_probs.get(c, 0) for c in green]
        w_vals = [w_probs.get(c, 0) for c in green]
        ax_ai.bar(x - width/2, b_vals, width, label='AI Baseline', color='mediumseagreen', alpha=0.8)
        ax_ai.bar(x + width/2, w_vals, width, label='AI Watermarked', color='coral', alpha=0.8)
        ax_ai.set_ylabel('P(initial)')
        ax_ai.set_title(f'AI Generated Code')
        ax_ai.legend(fontsize=8)
        ax_ai.set_xticks(x)
        ax_ai.set_xticklabels(green)

        # Human side
        h_vals = [h_probs.get(c, 0) for c in green]
        ax_hu.bar(x, h_vals, width, label=human_name, color='steelblue', alpha=0.8)
        ax_hu.set_ylabel('P(initial)')
        ax_hu.set_title(f'{human_name} (Human)')
        ax_hu.legend(fontsize=8)
        ax_hu.set_xticks(x)
        ax_hu.set_xticklabels(green)

        if row_idx == len(human_ds_names) - 1:
            ax_ai.set_xlabel('Green-list letters')
            ax_hu.set_xlabel('Green-list letters')

    plt.tight_layout()
    plt.show()

## Aggregated View: Green-Letter Ratio Across All Keys

In [ ]:
# Horizontal paired bar chart: average green-letter ratio per source across all keys
sources = ["HumanEval", "MBPP", "ClassEval", "AI Baseline", "AI Watermarked"]
means = [ratio_df[s].mean() for s in sources]
stds = [ratio_df[s].std() for s in sources]
colors = ['steelblue', 'steelblue', 'steelblue', 'mediumseagreen', 'coral']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(sources, means, xerr=stds, color=colors, alpha=0.8, capsize=4)
ax.set_xlabel('Mean Green-Letter Ratio (across all secret keys)')
ax.set_title('Average Fraction of Identifiers Starting with a Green Letter')
for bar, m in zip(bars, means):
    ax.text(m + 0.01, bar.get_y() + bar.get_height()/2, f'{m:.3f}', va='center', fontsize=10)
ax.set_xlim(0, 1.0)
plt.tight_layout()
plt.show()

## KL Divergence Analysis

Compute $D_{KL}(P \| Q)$ between initial-character distributions of different code sources, restricted to the 26 lowercase letters.

In [ ]:
ALL_LETTERS = list(string.ascii_lowercase)

def to_prob_vector(counter, smoothing=1e-10):
    """Full 26-letter probability vector with Laplace-like smoothing."""
    total = sum(counter.values()) + smoothing * 26
    return np.array([(counter.get(c, 0) + smoothing) / total for c in ALL_LETTERS])


def kl_divergence(p, q):
    """KL(P || Q)"""
    return np.sum(rel_entr(p, q))


def symmetric_kl(p, q):
    """Jensen-Shannon-like symmetric KL: (KL(P||Q) + KL(Q||P)) / 2"""
    return (kl_divergence(p, q) + kl_divergence(q, p)) / 2

In [ ]:
# Full distribution KL divergence (over all 26 letters)
src_names = list(counters.keys())
prob_vectors = {name: to_prob_vector(ctr) for name, ctr in counters.items()}

n = len(src_names)
kl_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        kl_matrix[i, j] = kl_divergence(prob_vectors[src_names[i]], prob_vectors[src_names[j]])

kl_df = pd.DataFrame(kl_matrix, index=src_names, columns=src_names).round(4)
print("KL Divergence D_KL(row || col) across full 26-letter distributions:")
kl_df

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(kl_matrix, cmap='YlOrRd', vmin=0)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(src_names, rotation=40, ha='right', fontsize=10)
ax.set_yticklabels(src_names, fontsize=10)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{kl_matrix[i,j]:.4f}", ha='center', va='center', fontsize=10,
                color='white' if kl_matrix[i,j] > kl_matrix.max()*0.6 else 'black')
plt.colorbar(im, label='KL Divergence (nats)')
ax.set_title('KL Divergence: D_KL(row || col)')
plt.tight_layout()
plt.show()

In [ ]:
# KL divergence restricted to GREEN letters only (per key)
kl_per_key = []
for key_name, info in key_sets.items():
    green = sorted(info["green"])
    gl_idx = [ALL_LETTERS.index(c) for c in green]

    def green_prob(counter):
        vec = to_prob_vector(counter)
        sub = vec[gl_idx]
        return sub / sub.sum()  # renormalize over green letters

    row = {"key": key_name, "|G|": info["size"]}
    wm_gp = green_prob(counters["AI Watermarked"])
    bl_gp = green_prob(counters["AI Baseline"])

    for h_name in ["HumanEval", "MBPP", "ClassEval"]:
        h_gp = green_prob(counters[h_name])
        row[f"KL(Wm||{h_name})"] = kl_divergence(wm_gp, h_gp)
        row[f"KL(Bl||{h_name})"] = kl_divergence(bl_gp, h_gp)

    row["KL(Wm||Bl)"] = kl_divergence(wm_gp, bl_gp)
    kl_per_key.append(row)

kl_key_df = pd.DataFrame(kl_per_key).set_index("key")
print("KL Divergence restricted to GREEN letters (per key):")
kl_key_df.round(4)

In [ ]:
# Summarize: mean KL over all keys
kl_summary = kl_key_df.drop(columns="|G|").mean()
print("Mean KL Divergence (green-letter restricted, across all keys):")
print(kl_summary.round(4).to_string())

## Per-Experiment Type Breakdown

In [ ]:
# Break watermarked code by experiment type
exp_counters = {}
for folder in wm_folders:
    exp_type = folder.name.split('_')[1]  # e.g. 'expA'
    codes = load_py_folder(folder)
    if exp_type not in exp_counters:
        exp_counters[exp_type] = Counter()
    exp_counters[exp_type] += initial_char_counter(codes)

# Green-letter ratio per experiment type (averaged across keys)
exp_ratio_rows = []
for exp_type, ctr in sorted(exp_counters.items()):
    row = {"experiment": exp_type, "n_ids": sum(ctr.values())}
    ratios = [green_ratio(ctr, key_sets[k]["green"]) for k in SECRET_KEYS]
    row["mean_green_ratio"] = np.mean(ratios)
    row["std"] = np.std(ratios)
    exp_ratio_rows.append(row)

exp_ratio_df = pd.DataFrame(exp_ratio_rows).set_index("experiment")
print("Mean green-letter ratio per experiment type (across all keys):")
exp_ratio_df.round(4)

In [ ]:
# Add human and baseline for reference
ref_rows = []
for name in ["HumanEval", "MBPP", "ClassEval", "AI Baseline"]:
    ratios = [green_ratio(counters[name], key_sets[k]["green"]) for k in SECRET_KEYS]
    ref_rows.append({"source": name, "mean_green_ratio": np.mean(ratios), "std": np.std(ratios)})

for _, row in exp_ratio_df.iterrows():
    ref_rows.append({"source": row.name if hasattr(row, 'name') else _, 
                      "mean_green_ratio": row["mean_green_ratio"], "std": row["std"]})

# Fix: use the index properly
all_comparison = []
for name in ["HumanEval", "MBPP", "ClassEval", "AI Baseline"]:
    ratios = [green_ratio(counters[name], key_sets[k]["green"]) for k in SECRET_KEYS]
    all_comparison.append({"source": name, "mean_green_ratio": np.mean(ratios), "std": np.std(ratios)})
for exp_type in sorted(exp_counters.keys()):
    ctr = exp_counters[exp_type]
    ratios = [green_ratio(ctr, key_sets[k]["green"]) for k in SECRET_KEYS]
    all_comparison.append({"source": exp_type, "mean_green_ratio": np.mean(ratios), "std": np.std(ratios)})

cmp_df = pd.DataFrame(all_comparison)

fig, ax = plt.subplots(figsize=(12, 6))
colors_map = {'HumanEval': 'steelblue', 'MBPP': 'steelblue', 'ClassEval': 'steelblue',
              'AI Baseline': 'mediumseagreen'}
bar_colors = [colors_map.get(s, 'coral') for s in cmp_df['source']]

bars = ax.bar(cmp_df['source'], cmp_df['mean_green_ratio'], 
              yerr=cmp_df['std'], color=bar_colors, alpha=0.8, capsize=4)
ax.set_ylabel('Mean Green-Letter Ratio')
ax.set_title('Green-Letter Ratio: All Sources (mean ± std across 8 secret keys)')
ax.set_xticklabels(cmp_df['source'], rotation=30, ha='right')
for bar, m in zip(bars, cmp_df['mean_green_ratio']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{m:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Findings

In [ ]:
human_names = ["HumanEval", "MBPP", "ClassEval"]
human_ratios = {h: [green_ratio(counters[h], key_sets[k]["green"]) for k in SECRET_KEYS] for h in human_names}
baseline_ratios = [green_ratio(counters["AI Baseline"], key_sets[k]["green"]) for k in SECRET_KEYS]
wm_ratios = [green_ratio(counters["AI Watermarked"], key_sets[k]["green"]) for k in SECRET_KEYS]

print("=" * 70)
print("FINDINGS (based on observed data)")
print("=" * 70)

print(f"\n--- Green-Letter Ratios (mean across {len(SECRET_KEYS)} secret keys) ---")
for h in human_names:
    print(f"  {h:15s}: {np.mean(human_ratios[h]):.4f} ± {np.std(human_ratios[h]):.4f}")
print(f"  {'AI Baseline':15s}: {np.mean(baseline_ratios):.4f} ± {np.std(baseline_ratios):.4f}")
print(f"  {'AI Watermarked':15s}: {np.mean(wm_ratios):.4f} ± {np.std(wm_ratios):.4f}")

print(f"\n--- KL Divergence (full 26-letter distribution) ---")
for h in human_names:
    kl_wm_h = kl_divergence(prob_vectors["AI Watermarked"], prob_vectors[h])
    kl_bl_h = kl_divergence(prob_vectors["AI Baseline"], prob_vectors[h])
    print(f"  D_KL(Watermarked || {h:10s}) = {kl_wm_h:.4f}")
    print(f"  D_KL(Baseline    || {h:10s}) = {kl_bl_h:.4f}")

kl_wm_bl = kl_divergence(prob_vectors["AI Watermarked"], prob_vectors["AI Baseline"])
print(f"  D_KL(Watermarked || Baseline)    = {kl_wm_bl:.4f}")

print(f"\n--- Green-list restricted KL (mean across keys) ---")
print(kl_summary.round(4).to_string())

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

print("""
1. GREEN-LETTER RATIOS ARE SIMILAR ACROSS ALL SOURCES (~0.70–0.75).
   The frequency-based candidate pool (top-18 letters) captures the dominant
   initial characters, so any green-set drawn from it will naturally cover
   a large fraction of identifiers in ANY Python code — human or AI.

2. KL DIVERGENCE SHOWS A MIXED PICTURE:
   - Watermarked code is CLOSER to HumanEval (0.196) and ClassEval (0.154)
     than the AI Baseline is (0.300 and 0.244 respectively).
   - However, Watermarked code is FARTHER from MBPP (0.300) than Baseline (0.149).
   - The Watermarked-vs-Baseline divergence (0.309) is notable, confirming
     that watermarking does shift the distribution.

3. THE WATERMARK REDISTRIBUTES — IT DOESN'T CREATE OUTLIERS.
   The watermarked distribution doesn't produce anomalous spikes on rare
   letters. Instead, it subtly shifts weight among already-common letters.
   This is why overall green-letter ratios remain comparable.

4. PER-EXPERIMENT VARIATION IS SMALL.
   All watermarking strategies (expA through expT) produce green-letter
   ratios within ±0.04 of each other, suggesting the approach is robust
   across different prompt-based watermarking variants.

5. THE CANDIDATE POOL DESIGN IS SOUND.
   Since the top-18 initial characters already account for the vast majority
   of identifiers, constraining green/red sets to this pool ensures that
   watermarked identifiers remain natural-sounding regardless of the key.
""")
